## Document translation

In [1]:
# load .env
import os
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.translation.document import DocumentTranslationClient
import os
endpoint = "https://brc-azure-translator.cognitiveservices.azure.com/"
key = os.getenv("AZURE_DOCUMENT_TRANSLATION_KEY")

document_translation_client = DocumentTranslationClient(endpoint, AzureKeyCredential(key))

In [ ]:
from azure.ai.translation.document import DocumentTranslationInput, TranslationTarget

import os

import time

def translate_document_with_azure_document_translation(

    document_translation_client,
    source_container_sas_url: str,
    target_container_sas_url: str,
    target_language: str = "en",
):
    """Translate documents using Azure Document Translation.
    NOTE: The Document Translation service works with Azure Blob Storage only.
    Both `source_container_sas_url` and `target_container_sas_url` must be
    SAS URLs pointing to Blob Storage containers or folders. It cannot
    translate a purely local file path without uploading it to Azure storage.
    """
    if not source_container_sas_url or not target_container_sas_url:
        raise ValueError(
            "Both source_container_sas_url and target_container_sas_url must be provided "
            "as SAS URLs to Azure Blob Storage containers."
        )
    translation_input = DocumentTranslationInput(
        source_url=source_container_sas_url,
        targets=[

            TranslationTarget(

                target_url=target_container_sas_url,

                language=target_language,

            )
        ],
    )

    poller = document_translation_client.begin_translation(inputs=[translation_input])
    # Wait for the translation operation to complete
    while not poller.done():
        time.sleep(5)

    result = poller.result()
    documents = []

    for doc in result.documents:

        documents.append(
            {
                "source_document_url": doc.source_document_url,
                "translated_document_url": doc.translated_document_url,
                "status": doc.status,
                "error": str(doc.error) if getattr(doc, "error", None) else None,
            }
        )

    return {"documents": documents}

# Example usage with SAS URLs from environment variables

source_container_sas_url = os.getenv("AZURE_TRANSLATION_SOURCE_CONTAINER_SAS_URL")

target_container_sas_url = os.getenv("AZURE_TRANSLATION_TARGET_CONTAINER_SAS_URL")


if source_container_sas_url and target_container_sas_url:

    result = translate_document_with_azure_document_translation(

        document_translation_client,

        source_container_sas_url,

        target_container_sas_url,

        target_language="en",

    )
    print(result)

HttpResponseError: (InvalidDocumentAccessLevel): Cannot access source document location with the current permissions.

In [ ]:
import os
import uuid
import json
import requests

translator_endpoint = os.getenv("AZURE_TRANSLATOR_TEXT_ENDPOINT", "https://api.cognitive.microsofttranslator.com")
translator_key = os.getenv("AZURE_TRANSLATOR_TEXT_KEY")
translator_region = os.getenv("AZURE_TRANSLATOR_TEXT_REGION", "canadacentral")


def translate_local_text_file_with_translator_text(file_path: str, target_language: str = "en"):

    """Translate the contents of a local text file using Azure Translator Text.
    This function does NOT upload the file itself anywhere.
    It only reads the file locally, then sends the extracted text
    to the Translator Text API for translation.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    if not translator_key:
        raise RuntimeError("AZURE_TRANSLATOR_TEXT_KEY is not set in the environment.")

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    path = "/translate"
    params = {

        "api-version": "3.0",

        "to": target_language,

    }

    constructed_url = translator_endpoint.rstrip("/") + path

    headers = {
        "Ocp-Apim-Subscription-Key": translator_key,
        "Content-type": "application/json",
        "X-ClientTraceId": str(uuid.uuid4()),
    }

    if translator_region:
        headers["Ocp-Apim-Subscription-Region"] = translator_region


    body = [
        {
            "text": content,
        }
    ]

    response = requests.post(constructed_url, params=params, headers=headers, json=body)
    response.raise_for_status()
    result = response.json()


    translations = []
    for item in result:
        for tr in item.get("translations", []):
            translations.append(tr.get("text"))

    return {
        "target_language": target_language,
        "translations": translations,
    }

# Example usage for a local text file (no upload):

local_file_path = ""
print(translate_local_text_file_with_translator_text(local_file_path, target_language="en"))


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

In [6]:
# Azure vision examples using Azure AI Vision API key

endpoint = "https://brc-azure-vision.cognitiveservices.azure.com/"
api_key = "your_api"

# sample usage
from azure.ai.vision import VisionClient, VisionServiceOptions, ImageSource, ImageAnalysisOptions, VisionFeature
_service_options = VisionServiceOptions(endpoint=endpoint, api_key=api_key)
client = VisionClient(service_options=_service_options)
image_path = "C:/Users/Admin/Documents/Border Clinic Project/BRC Sample Data/Client 2 - discrepancy/CNH Full.jpg"

image_source = ImageSource.from_file(image_path)
image_analysis_options = ImageAnalysisOptions(features=[VisionFeature.READ])
analysis = client.analyze_image(image_source, image_analysis_options)
print(analysis.read_results)


ImportError: cannot import name 'VisionClient' from 'azure.ai.vision' (unknown location)